# Food Image Classification with EfficientNetB3
### Transfer Learning on the `bjoernjostein/food-classification` Dataset (61 Classes)

## Prerequisites

Before running this notebook, ensure you have Kaggle API credentials configured:
1. Go to https://www.kaggle.com/settings/api
2. Click **Create New Token** — this downloads `kaggle.json`
3. Place `kaggle.json` at `~/.kaggle/kaggle.json` (Linux/Mac) or `C:\\Users\\<you>\\.kaggle\\kaggle.json` (Windows)
4. On Linux/Mac: `chmod 600 ~/.kaggle/kaggle.json`

The notebook uses `kagglehub` which reads these credentials automatically.

In [ ]:
!pip install kagglehub timm torch torchvision numpy pandas matplotlib seaborn scikit-learn Pillow tqdm --quiet

## Section 1 — Setup & Imports

In [ ]:
import os
import random
import warnings
import pathlib

import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, cohen_kappa_score
)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
import torchvision.transforms as T
import timm

warnings.filterwarnings('ignore')

In [ ]:
SEED         = 42
DATASET_SLUG = "bjoernjostein/food-classification"
IMG_SIZE     = (224, 224)
BATCH_SIZE   = 32
EPOCHS       = 5 if not torch.cuda.is_available() else 20
LR           = 1e-4
FINE_TUNE_LR = 1e-5
NUM_WORKERS  = 0   # 0 is safest on Windows; increase if on Linux
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Config: EPOCHS={EPOCHS}, BATCH_SIZE={BATCH_SIZE}, DEVICE={DEVICE}")
if not torch.cuda.is_available():
    print("⚠️  No GPU detected. Training will be slow. Consider using Google Colab or reducing EPOCHS.")
else:
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")

In [ ]:
def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
os.makedirs("figures", exist_ok=True)
print("Seed set. figures/ directory ready.")

## Section 2 — Data Download with kagglehub

In [ ]:
raw_path = kagglehub.dataset_download(DATASET_SLUG)
DATA_ROOT = pathlib.Path(raw_path)
print(f"Dataset downloaded to: {DATA_ROOT}")

In [ ]:
for root, dirs, files in os.walk(DATA_ROOT):
    level = str(root).replace(str(DATA_ROOT), '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{pathlib.Path(root).name}/')
    if level < 2:
        for f in files[:5]:
            print(f'{indent}  {f}')

In [ ]:
_IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

def find_split_dir(root: pathlib.Path, candidates: list) -> pathlib.Path | None:
    for name in candidates:
        p = root / name
        if p.is_dir():
            return p
    return None

TRAIN_DIR = find_split_dir(DATA_ROOT, ["train", "Train", "training"])
VAL_DIR   = find_split_dir(DATA_ROOT, ["valid", "validation", "val", "Valid"])
TEST_DIR  = find_split_dir(DATA_ROOT, ["test", "Test"])

assert TRAIN_DIR is not None, "Could not find a train/ directory in the dataset"
print(f"Train dir : {TRAIN_DIR}")
print(f"Val dir   : {VAL_DIR}")
print(f"Test dir  : {TEST_DIR}")

CLASS_NAMES = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
print(f"
Found {NUM_CLASSES} classes: {CLASS_NAMES[:5]} ...")

In [ ]:
def count_images(split_dir: pathlib.Path) -> dict:
    """Return {class_name: count} for a split directory."""
    counts = {}
    for cls_dir in sorted(split_dir.iterdir()):
        if cls_dir.is_dir():
            counts[cls_dir.name] = sum(
                1 for f in cls_dir.iterdir()
                if f.suffix.lower() in _IMAGE_EXTS
            )
    return counts

train_counts = count_images(TRAIN_DIR)
val_counts   = count_images(VAL_DIR) if VAL_DIR else {}
test_counts  = count_images(TEST_DIR) if TEST_DIR else {}

print(f"Train images : {sum(train_counts.values())}")
print(f"Val images   : {sum(val_counts.values())}")
print(f"Test images  : {sum(test_counts.values())}")

In [ ]:
# Build full training file list
_all_files, _all_labels = [], []
for cls_name in CLASS_NAMES:
    cls_dir = TRAIN_DIR / cls_name
    for f in cls_dir.iterdir():
        if f.suffix.lower() in _IMAGE_EXTS:
            _all_files.append(str(f))
            _all_labels.append(CLASS_TO_IDX[cls_name])

# Carve test set from training data if no test dir exists
if TEST_DIR is None:
    print("No test/ directory found — carving 15% of training data as test set (stratified).")
    train_files, test_files, train_lbls, test_lbls = train_test_split(
        _all_files, _all_labels, test_size=0.15, random_state=SEED, stratify=_all_labels
    )
else:
    train_files, train_lbls = _all_files, _all_labels
    test_files, test_lbls = [], []
    for cls_name in CLASS_NAMES:
        test_cls = TEST_DIR / cls_name
        if test_cls.is_dir():
            for f in test_cls.iterdir():
                if f.suffix.lower() in _IMAGE_EXTS:
                    test_files.append(str(f))
                    test_lbls.append(CLASS_TO_IDX[cls_name])

# Carve val set from training data if no val dir exists
if VAL_DIR is None:
    print("No val/ directory found — carving 15% of training data as validation set (stratified).")
    train_files, val_files, train_lbls, val_lbls = train_test_split(
        train_files, train_lbls, test_size=0.15, random_state=SEED, stratify=train_lbls
    )
else:
    val_files, val_lbls = [], []
    for cls_name in CLASS_NAMES:
        val_cls = VAL_DIR / cls_name
        if val_cls.is_dir():
            for f in val_cls.iterdir():
                if f.suffix.lower() in _IMAGE_EXTS:
                    val_files.append(str(f))
                    val_lbls.append(CLASS_TO_IDX[cls_name])

print(f"
Final split sizes:")
print(f"  Train : {len(train_files)}")
print(f"  Val   : {len(val_files)}")
print(f"  Test  : {len(test_files)}")